# Финальный блок выделения `gold zones` для ML-прогноза

Этот блок нужен вставить **после расчёта итогового поля перспективности** (`prospectivity`, `gb_score`, `ml_score` и т.п.).

Логика не использует ручные бонусы. Жёлтые зоны выделяются из ML-поля через устойчивый двухпороговый отбор:

1. **candidate zone** — перспективные ячейки верхнего квантиля;
2. **core zone** — самые уверенные ячейки ещё более высокого квантиля;
3. сохраняются только те candidate-кластеры, внутри которых есть core;
4. мелкий шум удаляется по площади / числу ячеек.

Так зоны не «расползаются» и не превращают всё синее поле в жёлтое.


In [ ]:
# ============================================================
# FINAL GOLD ZONES SELECTION FOR ML PROSPECTIVITY MAP
# ============================================================
# Требования:
#   grid : GeoDataFrame с ячейками прогноза
#   score_col : колонка с итоговым ML-прогнозом, где БОЛЬШЕ = перспективнее
#
# Результат:
#   grid["gold_zone_ml"]      -> True/False для жёлтых зон
#   grid["gold_cluster"]      -> номер кластера
#   gold_zones_ml             -> GeoDataFrame с полигонами жёлтых зон
# ============================================================

import numpy as np
import geopandas as gpd
from scipy.ndimage import label, binary_closing, binary_opening


def _infer_regular_grid_indices(gdf, cell_size=None):
    """
    Восстанавливает row/col для регулярной сетки по центроидам.
    Работает для обычной квадратной сетки 500x500 м.
    """
    cent = gdf.geometry.centroid
    x = cent.x.to_numpy()
    y = cent.y.to_numpy()

    if cell_size is None:
        xs = np.sort(np.unique(np.round(x, 6)))
        ys = np.sort(np.unique(np.round(y, 6)))

        dx = np.diff(xs)
        dy = np.diff(ys)
        dx = dx[dx > 0]
        dy = dy[dy > 0]

        if len(dx) == 0 or len(dy) == 0:
            cell_size = 500.0
        else:
            cell_size = float(np.median(np.r_[dx, dy]))

    col = np.round((x - np.nanmin(x)) / cell_size).astype(int)
    # row делаем сверху вниз, чтобы массив нормально рисовался через imshow
    row = np.round((np.nanmax(y) - y) / cell_size).astype(int)
    return row, col, cell_size


def select_gold_zones_from_ml(
    grid,
    score_col="prospectivity",
    mask_col=None,
    candidate_q=0.90,
    core_q=0.965,
    min_cluster_cells=8,
    max_cluster_cells=None,
    cell_size=None,
    closing_iterations=1,
    opening_iterations=0,
    invert_score=False,
    output_col="gold_zone_ml",
    cluster_col="gold_cluster",
):
    """
    Устойчивое выделение компактных перспективных зон из ML-карты.

    Параметры, которые реально стоит менять:
    - candidate_q:
        ниже -> жёлтых зон больше;
        выше -> зон меньше.
        Хороший диапазон: 0.88–0.93.

    - core_q:
        отвечает за наличие уверенного ядра внутри зоны.
        Хороший диапазон: 0.955–0.98.

    - min_cluster_cells:
        удаляет мелкие случайные пятна.
        Для сетки 500 м: 6–15 ячеек обычно нормально.

    - invert_score:
        False, если большее значение score = лучше.
        True, если используется поле типа prognoz, где меньше = лучше.
    """
    if score_col not in grid.columns:
        raise KeyError(f"В grid нет колонки '{score_col}'. Доступные колонки: {list(grid.columns)}")

    g = grid.copy()
    score = g[score_col].astype(float).to_numpy()

    if invert_score:
        # переводим поле "меньше = лучше" в "больше = лучше"
        score = np.nanmax(score) - score

    # область анализа: только валидные ячейки и, если есть, маска прогноза
    valid = np.isfinite(score)
    if mask_col is not None and mask_col in g.columns:
        valid &= g[mask_col].fillna(0).astype(float).to_numpy() > 0

    if valid.sum() == 0:
        raise ValueError("Нет валидных ячеек для выделения gold zones. Проверь score_col и mask_col.")

    # нормализация только внутри области прогноза
    s = score.copy()
    lo, hi = np.nanpercentile(s[valid], [1, 99])
    if hi <= lo:
        lo, hi = np.nanmin(s[valid]), np.nanmax(s[valid])
    s_norm = np.clip((s - lo) / (hi - lo + 1e-12), 0, 1)

    row, col, used_cell_size = _infer_regular_grid_indices(g, cell_size=cell_size)
    nrows = int(row.max() + 1)
    ncols = int(col.max() + 1)

    score_arr = np.full((nrows, ncols), np.nan, dtype=float)
    valid_arr = np.zeros((nrows, ncols), dtype=bool)
    index_arr = np.full((nrows, ncols), -1, dtype=int)

    for i, (r, c) in enumerate(zip(row, col)):
        score_arr[r, c] = s_norm[i]
        valid_arr[r, c] = valid[i]
        index_arr[r, c] = i

    vals = score_arr[valid_arr]
    candidate_thr = float(np.nanquantile(vals, candidate_q))
    core_thr = float(np.nanquantile(vals, core_q))

    candidate = (score_arr >= candidate_thr) & valid_arr
    core = (score_arr >= core_thr) & valid_arr

    # Морфология нужна только для карты: закрывает дырки, но не должна раздувать зоны сильно.
    if closing_iterations and closing_iterations > 0:
        candidate = binary_closing(candidate, iterations=closing_iterations)
        candidate &= valid_arr

    if opening_iterations and opening_iterations > 0:
        candidate = binary_opening(candidate, iterations=opening_iterations)
        candidate &= valid_arr

    # 8-связность: диагонально соседние ячейки считаются одним кластером
    structure = np.ones((3, 3), dtype=int)
    labels, nlab = label(candidate, structure=structure)

    keep_arr = np.zeros_like(candidate, dtype=bool)
    cluster_arr = np.zeros_like(labels, dtype=int)

    new_id = 1
    for lab_id in range(1, nlab + 1):
        comp = labels == lab_id
        size = int(comp.sum())

        # кластер должен быть не слишком маленьким
        if size < min_cluster_cells:
            continue

        # опционально можно отсечь гигантские полосы
        if max_cluster_cells is not None and size > max_cluster_cells:
            continue

        # главный стабилизатор: зона должна содержать уверенное ядро
        if not np.any(comp & core):
            continue

        keep_arr |= comp
        cluster_arr[comp] = new_id
        new_id += 1

    gold_flag = np.zeros(len(g), dtype=bool)
    gold_cluster = np.zeros(len(g), dtype=int)

    rr, cc = np.where(keep_arr)
    for r, c in zip(rr, cc):
        i = index_arr[r, c]
        if i >= 0:
            gold_flag[i] = True
            gold_cluster[i] = int(cluster_arr[r, c])

    g[output_col] = gold_flag
    g[cluster_col] = gold_cluster
    g["prospectivity_norm"] = s_norm

    gold_cells = g[g[output_col]].copy()
    if len(gold_cells) > 0:
        gold_zones = gold_cells.dissolve(by=cluster_col, as_index=False)
        gold_zones["zone_area_km2"] = gold_zones.geometry.area / 1_000_000
        gold_zones["mean_score"] = gold_cells.groupby(cluster_col)["prospectivity_norm"].mean().values
        gold_zones["max_score"] = gold_cells.groupby(cluster_col)["prospectivity_norm"].max().values
    else:
        gold_zones = gpd.GeoDataFrame(columns=[cluster_col, "zone_area_km2", "mean_score", "max_score", "geometry"], crs=g.crs)

    print("Gold zones selection:")
    print(f"  score_col           = {score_col}")
    print(f"  cell_size           = {used_cell_size:.2f}")
    print(f"  candidate_q         = {candidate_q}")
    print(f"  core_q              = {core_q}")
    print(f"  candidate_threshold = {candidate_thr:.3f}")
    print(f"  core_threshold      = {core_thr:.3f}")
    print(f"  selected_cells      = {int(gold_flag.sum())}")
    print(f"  selected_clusters   = {len(gold_zones)}")

    return g, gold_zones


# ============================================================
# ЗАПУСК
# ============================================================
# ВАЖНО:
# - если твоя итоговая колонка называется иначе, замени score_col.
# - если есть маска прогноза sed, лучше оставить mask_col="sed".
# - если используешь поле prognoz, где МЕНЬШЕ = перспективнее, поставь invert_score=True.

score_col = "prospectivity"  # варианты: "prospectivity", "gb_score", "ml_score", "prognoz"
mask_col = "sed" if "sed" in grid.columns else None

# Рекомендуемые настройки для твоей карты:
# candidate_q=0.91..0.93 — если жёлтого слишком много, увеличь.
# core_q=0.965..0.98 — если зоны плывут, увеличь.
grid, gold_zones_ml = select_gold_zones_from_ml(
    grid,
    score_col=score_col,
    mask_col=mask_col,
    candidate_q=0.92,
    core_q=0.975,
    min_cluster_cells=6,
    max_cluster_cells=160,   # защищает от длинных полос по разломам; можно None
    closing_iterations=1,
    opening_iterations=0,
    invert_score=False,
    output_col="gold_zone_ml",
    cluster_col="gold_cluster",
)

# Если после запуска золота всё ещё много:
#   candidate_q=0.94, core_q=0.982
# Если золота слишком мало:
#   candidate_q=0.90, core_q=0.965


In [ ]:
# ============================================================
# QUICK PLOT CHECK
# ============================================================
# Этот блок только для быстрой проверки результата.
# Он не влияет на модель.

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 9))

grid.plot(
    column="prospectivity_norm",
    ax=ax,
    cmap="bwr_r",       # синий = перспективнее, красный = хуже
    linewidth=0,
    legend=False,
)

# контуры исходных объектов/геологии, если у тебя есть общий слой boundaries/contours — можно заменить
if "geometry" in grid.columns:
    try:
        grid.boundary.plot(ax=ax, color="black", linewidth=0.05, alpha=0.15)
    except Exception:
        pass

if len(gold_zones_ml) > 0:
    gold_zones_ml.plot(
        ax=ax,
        color="yellow",
        edgecolor="black",
        linewidth=0.8,
        label="Gold zones ML",
    )

ax.set_title("Итоговый ML-прогноз: компактные gold zones", fontsize=12)
ax.set_axis_off()
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# SAVE RESULT
# ============================================================
# Сохранение результата в GeoPackage.

from pathlib import Path

out_dir = Path("gb_final_result")
out_dir.mkdir(exist_ok=True)

grid.to_file(out_dir / "grid_with_ml_gold_zones.gpkg", layer="grid", driver="GPKG")

if len(gold_zones_ml) > 0:
    gold_zones_ml.to_file(out_dir / "ml_gold_zones.gpkg", layer="gold_zones", driver="GPKG")

print("Saved to:", out_dir.resolve())


## Как подбирать параметры

Для твоей ситуации я бы начинал с таких значений:

```python
candidate_q=0.92
core_q=0.975
min_cluster_cells=6
max_cluster_cells=160
closing_iterations=1
```

Если жёлтые зоны слишком большие — подними `candidate_q` до `0.93–0.94` и `core_q` до `0.98`.

Если жёлтых зон почти нет — опусти `candidate_q` до `0.90–0.91`.

Если появляются длинные полосы вдоль разломов — уменьши `max_cluster_cells`, например до `100–130`.

Если нужная зона разорвана на куски — увеличь `closing_iterations` до `2`, но осторожно: это может снова раздуть жёлтые области.
